In [0]:
import pandas as pd

li_data = [
    'addresses',
    'claims',
    'customers',
    'policies_buildings',
    'policies_contents',
    'policies_motor',
    'sample_submission',
    'test_buildings',
    'test_contents',
    'test_motor',
    'weather'
]

kw_data = {}  

for fname in li_data:
    kw_data[fname] = pd.read_parquet(f'../data/{fname}.parquet')

In [0]:
import os
os.listdir('../data')


In [0]:
# Preview the first rows of selected datasets
for name in ['claims', 'policies_buildings', 'sample_submission', 'test_buildings']:
    df = kw_data[name]
    print(f'{name}: {df.shape[0]:,} rows x {df.shape[1]} columns')
    display(df.head(5))

### Initial data observations

| Dataset | Rows | Columns |
| --- | --- | --- |
| claims | 163,150 | 12 |
| policies_buildings | 707,726 | 27 |
| sample_submission | 626,639 | 2 |
| test_buildings | 184,155 | 26 |

* **Target = claim prediction**: the `prediction` column in the submission is the expected **ultimate claim cost** per row (all claims with a `loss_date` in the row's period, net of excess and recoveries, excluding repudiated claims). This is the risk premium, not the premium charged, and it is scored with Tweedie D² (power 1.5). It has to be built from the `claims` file, because there is no target column in the policy files.
* **`renewal_decision` is not the target**: it is in the training files only (`test_buildings` has every other `policies_buildings` column). It is known only after the period, so exclude it from features to avoid leakage.
* **Submission covers all three lines**: `sample_submission` rows = test buildings + contents + motor (184,155 + 187,498 + 254,986 = 626,639). The sample row_ids start with `M` (motor) and `prediction` is numeric.
* **`claims` spans multiple lines**: it has a `line` column (e.g. `motor`) and joins to policies on `policy_id`, so filter to buildings claims before joining.
* **`construction_type` needs cleaning**: inconsistent labels such as `Brick`, `stone`, `T`, `W`, `NON_STANDARD`.
* **Same policies recur across years**: e.g. `B89645301` appears in 2021 training data and 2025 test data, so watch for leakage when building validation splits.

### Checking the `policy_id` prefix

The first letter of `policy_id` appears to encode the product line (e.g. `M` = motor, `B` = buildings). Before relying on this, the cell below:

* Lists which raw datasets contain a `policy_id` column (`addresses`, `customers`, `weather` and `sample_submission` do not).
* Counts the first letter of `policy_id` in each dataset. Each `policies_*` and `test_*` file should contain only one letter.
* Cross-tabulates the prefix against the `line` column in `claims` as an independent check: if the prefix encodes the line, each letter should map to exactly one `line` value.

In [0]:
# Count first letter of policy_id in every dataset that has the column
prefix_summary = []
for name, df in kw_data.items():
    if 'policy_id' not in df.columns:
        print(f'{name}: no policy_id column')
        continue
    counts = df['policy_id'].str[0].value_counts(dropna=False)
    for prefix, n in counts.items():
        prefix_summary.append({'dataset': name, 'prefix': prefix, 'rows': n})

display(pd.DataFrame(prefix_summary))

# Independent check: prefix vs the claims 'line' column
display(pd.crosstab(kw_data['claims']['policy_id'].str[0], kw_data['claims']['line']).reset_index())

### Adding a `policy_type` column

The check above confirms the prefix maps one-to-one to the product line:

| Prefix | policy_type |
| --- | --- |
| `B` | buildings |
| `C` | contents |
| `M` | motor |

The cell below builds a new dictionary, `kw_data_typed`, with a copy of every raw dataset. The originals in `kw_data` are not changed. For each dataset that has a `policy_id` column, it adds `policy_type` by mapping the first letter of `policy_id` with the table above. Datasets without `policy_id` are copied unchanged. As a safeguard, the cell stops with an error if any `policy_id` has a prefix that is not in the mapping.

In [0]:
# Map policy_id prefix to product line
policy_type_map = {'B': 'buildings', 'C': 'contents', 'M': 'motor'}

kw_data_typed = {}
for name, df in kw_data.items():
    df_new = df.copy()
    if 'policy_id' in df_new.columns:
        df_new['policy_type'] = df_new['policy_id'].str[0].map(policy_type_map)
        n_unmapped = df_new['policy_type'].isna().sum()
        assert n_unmapped == 0, f'{name}: {n_unmapped} rows with unknown policy_id prefix'
    kw_data_typed[name] = df_new

# Summary of policy_type per dataset
summary = [
    {'dataset': name, 'policy_type': ptype, 'rows': n}
    for name, df in kw_data_typed.items() if 'policy_type' in df.columns
    for ptype, n in df['policy_type'].value_counts().items()
]
display(pd.DataFrame(summary))

### Adding an `earned_premium` column

`written_premium` is the full premium charged for the policy period. `exposure` is the fraction of a year the policy was actually in force (e.g. `1.0` = a full year, `0.3452` = about 4 months). Multiplying the two gives the premium **earned** over the time the policy was in force:

`earned_premium = exposure * written_premium`

The cell below loops over every dataset in `kw_data_typed` that has an `exposure` column and adds `earned_premium`. If a dataset has `exposure` but no `written_premium`, it is skipped and listed so it can be checked. The summary table shows total exposure, written premium and earned premium per dataset as a sense check: earned premium should never be more than written premium when exposure is at most 1.

In [0]:
# Add earned_premium = exposure * written_premium wherever exposure exists
ep_summary = []
for name, df in kw_data_typed.items():
    if 'exposure' not in df.columns:
        continue
    if 'written_premium' not in df.columns:
        print(f'{name}: has exposure but no written_premium - skipped')
        continue
    df['earned_premium'] = df['exposure'] * df['written_premium']
    ep_summary.append({
        'dataset': name,
        'rows': len(df),
        'exposure_min': df['exposure'].min(),
        'exposure_max': df['exposure'].max(),
        'total_written_premium': round(df['written_premium'].sum(), 2),
        'total_earned_premium': round(df['earned_premium'].sum(), 2),
        'null_earned_premium': int(df['earned_premium'].isna().sum()),
    })

display(pd.DataFrame(ep_summary))

### Data quality note: `exposure` greater than 1 in training data

The sense check above fails for the training files: total `earned_premium` is **higher** than total `written_premium`, because `exposure` goes up to 365 there. The test files are fine (max exposure = 1).

Comparing `exposure` with the policy period length (`days = period_end - period_start + 1`) shows that every training row uses one of three conventions:

| Convention | How to recognise it | Example (full-year policy) |
| --- | --- | --- |
| **Years, 365-day basis** (the standard, and all test rows) | `exposure = days / 365` | `1.0` |
| **Days instead of years** | `exposure = days` | `365.0` |
| **Years, 360-day basis** | `exposure = days / 360` | `1.0139` (= 365/360) |

* About 0.4% of training rows have `exposure > 1`: 3,044 buildings, 3,088 contents and 3,991 motor rows. These are a mix of the day-count rows (e.g. `365.0`, `327.0`) and 360-day-basis full-year rows (`1.0139`).
* The problem is **not only** rows above 1: a further 1,138 buildings, 1,222 contents and 1,745 motor rows are on the 360-day basis with short periods, so their exposure is at most 1 but still about 1.4% too high. A simple "cap at 1" would miss these.
* In total, the training rows needing conversion are:

| Dataset | Stored in days | 360-day basis | Already 365-day basis |
| --- | --- | --- | --- |
| policies_buildings | 656 | 3,526 | 703,544 |
| policies_contents | 729 | 3,582 | 717,306 |
| policies_motor | 934 | 4,802 | 974,232 |

* One contents row is a 1-day policy stored in days (`exposure = 1`), which looks like a valid full year.
* No `exposure` values are null, and period lengths range from 1 to 365 days.

The raw `earned_premium` in the cell above overstates premium for these rows. The next cells standardise exposure to the 365-day year basis and recalculate `earned_premium`.

### Standardising exposure and recalculating `earned_premium`

The cell below adds `exposure_standardised` (fraction of a 365-day year) to every dataset that has `exposure`, `period_start` and `period_end`:

1. Calculate `days = period_end - period_start + 1`.
2. Classify each row's convention, checked in this order:
   * `year_365`: `exposure` matches `days / 365` (within 4-decimal rounding). Kept as is.
   * `days`: `exposure` equals `days` (and is at least 1). Converted to `days / 365`.
   * `year_360`: `exposure` matches `days / 360`. Converted to `days / 365`.

   Converted rows use `days / 365` directly rather than rescaling the stored value, because the stored 360-basis values are rounded to 4 decimals (e.g. `1.0139 * 360 / 365` gives 1.00001, not exactly 1).
3. Stop with an error if any row matches none of these, so unexpected values are never silently converted.
4. Overwrite `earned_premium` with `exposure_standardised * written_premium`. The raw `exposure` column is kept unchanged.

The summary table shows how many rows were converted from each convention, and checks that `exposure_standardised` is now between 0 and 1 and that total `earned_premium` is no longer more than total `written_premium`.

In [0]:
import numpy as np

TOL = 0.00006  # exposure is stored to 4 decimal places

std_summary = []
for name, df in kw_data_typed.items():
    if not {'exposure', 'period_start', 'period_end'}.issubset(df.columns):
        continue

    days = (pd.to_datetime(df['period_end']) - pd.to_datetime(df['period_start'])).dt.days + 1
    exp = df['exposure']

    is_365 = np.isclose(exp, days / 365, atol=TOL, rtol=0)
    is_days = ~is_365 & (exp >= 1) & np.isclose(exp, days, atol=0.5, rtol=0)
    is_360 = ~is_365 & ~is_days & np.isclose(exp, days / 360, atol=TOL, rtol=0)
    unmatched = ~(is_365 | is_days | is_360)
    assert unmatched.sum() == 0, f'{name}: {unmatched.sum()} rows with unrecognised exposure'

    df['exposure_standardised'] = np.select(
        [is_365, is_days, is_360],
        [exp, days / 365, days / 365],  # converted rows use the exact period length
    )
    df['earned_premium'] = df['exposure_standardised'] * df['written_premium']

    std_summary.append({
        'dataset': name,
        'rows': len(df),
        'year_365_rows': int(is_365.sum()),
        'days_rows': int(is_days.sum()),
        'year_360_rows': int(is_360.sum()),
        'exposure_std_min': df['exposure_standardised'].min(),
        'exposure_std_max': df['exposure_standardised'].max(),
        'total_written_premium': round(df['written_premium'].sum(), 2),
        'total_earned_premium': round(df['earned_premium'].sum(), 2),
    })

display(pd.DataFrame(std_summary))

## Roadmap: buildings claim cost model

Each step is tracked as a GitHub issue on the project Kanban board. Work through them in order, and reference the issue in the commit message (e.g. `Aggregate claims per row - closes #15`) so it closes when pushed.

**Target reminder** (from `docs/kaggle_overview.md`): predict the expected **ultimate claim cost** for each 2025 row: all claims with a `loss_date` in the row's period, net of excess and recoveries, excluding repudiated claims. Scored with **Tweedie D², power 1.5**.

### Done in this notebook (pending commit)

| Issue | Step | Cells |
| --- | --- | --- |
| #10 | Fix data loading paths for Databricks | 1-2 |
| #11 | Initial data preview and observations | 4-5 |
| #12 | Derive `policy_type` from the `policy_id` prefix | 6-9 |
| #13 | Add `earned_premium` | 10-11, 14 |
| #14 | Standardise exposure to a 365-day basis | 12-14 |

### Next steps

1. **#16 Develop claims to ultimate (chain ladder).** Claims are valued at 31 Dec 2024, so recent years, especially 2024, are under-reported (IBNR) and open claims may still develop (IBNER). Estimate development factors from reporting delay and incurred development, then add a developed column (e.g. `incurred_ultimate`) to `claims` **before** aggregating. Without this, the model would learn that recent years have lower claims than they really do.
2. **#15 Aggregate claims per policy row.** A `policy_id` spans several years and can have several rows per year, so match each claim to the policy row whose `period_start` to `period_end` contains its `loss_date`. Then, per row, calculate `claim_count`, `total_paid`, `total_incurred` and `total_recoveries`, and the target net of recoveries excluding repudiated claims. Check that totals reconcile to the raw claims.
3. **#17 Join to buildings policies.** Left-join the aggregates onto `policies_buildings` by `row_id`, filling rows with no claims with 0. Check the row count is unchanged (707,726) and that frequency and loss ratio (vs `earned_premium`) by year look reasonable. Optionally add `addresses`, `weather` and `customers` features.
4. **#18 Fit XGBoost (buildings).** Use `reg:tweedie` with `tweedie_variance_power = 1.5` to match the scoring metric, with `log(exposure_standardised)` as an offset. Validate on a time-based split (e.g. train 2021-2023, validate 2024), because policies recur across years. Exclude leaking columns (`renewal_decision`, IDs), clean `construction_type`, and compare against a constant-rate baseline.
5. **#19 Buildings-only submission.** Score `test_buildings`, merge onto `sample_submission` by `row_id`, and set motor (`M`) and contents (`C`) rows to 0. Check for 626,639 unique rows, no nulls and no negative predictions. Note that zero predictions for motor and contents will score poorly on those rows, so this is a first buildings-only check, not the final submission.

## Submission v001: buildings constant-rate baseline

A first end-to-end submission, made before the claims work (#15-#17) and the XGBoost model (#18) are done. It checks the file format and gives a leaderboard reference point.

* **Buildings rows**: a single burning-cost rate per year of exposure, multiplied by each test row's `exposure_standardised`.
  * Rate = total buildings claim cost / total buildings `exposure_standardised` (2021-2024 training data).
  * Claim cost = `incurred_to_date - recoveries`, excluding repudiated claims, with no chain ladder development (#16). This **understates** cost, especially for 2024.
* **Motor and contents rows**: 0, as agreed for the buildings-only submission (#19).
* A constant rate is the Tweedie D² baseline, so the buildings rows should score about 0. Any gain from later versions comes from the model.

The file is saved to `submissions/home_insurance_submission_v001.csv` in the repo, with header `row_id,prediction`. The cell checks for 626,639 unique `row_id`s that match `sample_submission`, no nulls and no negative predictions.

In [0]:
import os

sub_template = kw_data_typed['sample_submission']
test_b = kw_data_typed['test_buildings']
pol_b = kw_data_typed['policies_buildings']
claims = kw_data_typed['claims']

# Share of the submission that is buildings (by row_id prefix)
line_share = (sub_template['row_id'].str[0].map(policy_type_map)
              .value_counts().rename('rows').to_frame())
line_share['pct'] = (line_share['rows'] / len(sub_template) * 100).round(2)
display(line_share.reset_index())

# Buildings burning cost per year of exposure (no IBNR/IBNER development yet)
b_claims = claims[(claims['policy_type'] == 'buildings') & (claims['status'] != 'repudiated')]
b_claim_cost = (b_claims['incurred_to_date'] - b_claims['recoveries']).sum()
b_exposure = pol_b['exposure_standardised'].sum()
b_rate = b_claim_cost / b_exposure
print(f'Buildings claim cost: {b_claim_cost:,.0f} CU | exposure: {b_exposure:,.0f} years | rate: {b_rate:,.2f} CU per year')

# Predictions: buildings = rate * exposure, everything else = 0
b_pred = test_b[['row_id']].assign(prediction=test_b['exposure_standardised'] * b_rate)
submission = (sub_template[['row_id']]
              .merge(b_pred, on='row_id', how='left')
              .fillna({'prediction': 0.0}))
# Round prediction to 2 decimal places (stays float64)
submission['prediction'] = submission['prediction'].astype('float64').round(2)

# Checks
assert len(submission) == len(sub_template) == 626_639
assert submission['row_id'].is_unique
assert set(submission['row_id']) == set(sub_template['row_id'])
assert submission['prediction'].notna().all() and (submission['prediction'] >= 0).all()
assert (submission.loc[submission['row_id'].str[0] != 'B', 'prediction'] == 0).all()
assert submission['row_id'].isin(test_b['row_id']).sum() == len(test_b)

# Save
out_dir = '../submissions'
os.makedirs(out_dir, exist_ok=True)
out_path = f'{out_dir}/home_insurance_submission_v001.csv'
# Write prediction with exactly 2 decimal places, no index
submission.to_csv(out_path, index=False, float_format='%.2f')

print(f'Saved {len(submission):,} rows to {os.path.abspath(out_path)}')
display(submission.groupby(submission['row_id'].str[0].map(policy_type_map))['prediction']
        .agg(['count', 'sum', 'mean']).reset_index())

### Submission v001: results

**Buildings is 29.39% of the submission file** (184,155 of 626,639 rows, by `row_id` prefix):

| Line | Rows | % of submission | v001 prediction |
| --- | --- | --- | --- |
| Motor (`M`) | 254,986 | 40.69% | 0 |
| Contents (`C`) | 187,498 | 29.92% | 0 |
| Buildings (`B`) | 184,155 | 29.39% | rate × exposure |

So v001 gives a real prediction for under a third of rows. The other ~70.6% (motor and contents) are set to 0 and will score poorly. Motor is the largest line, so it should be the priority after buildings.

* **Buildings rate**: 248.2M CU claim cost / 590,560 exposure years = **420.28 CU per policy year**, giving a mean prediction of 342.89 CU per test row (many 2025 rows cover part of a year). Total buildings prediction is 63.1M CU.
* **Sense check**: the rate is above buildings earned premium per exposure year (191.0M / 590,560 = about 323 CU), a loss ratio of about 130%. This needs reviewing alongside the chain ladder work (#16) and the claims aggregation (#15).
* **File checks passed**: 626,639 unique `row_id`s matching `sample_submission`, header `row_id,prediction`, no nulls, no negatives, and non-buildings rows are exactly 0. Re-reading the saved CSV gave the same totals.
* **Saved to**: `submissions/home_insurance_submission_v001.csv` (repo root).

### Submission v001: parquet version and format check

The cell below saves the same submission as `submissions/home_insurance_submission_v001.parquet`. It then reads the parquet file back and compares its **structure** with `data/sample_submission.parquet`:

* Same parquet schema (column names, order and Arrow types)
* Same row count and the same `row_id`s **in the same order**
* Same pandas dtypes
* No nulls or negative predictions

The `prediction` values are expected to differ, because the sample is all zeros. The cell also confirms that the parquet and CSV versions contain identical data.

In [0]:
import pyarrow.parquet as pq

sample_path = '../data/sample_submission.parquet'
parquet_path = f'{out_dir}/home_insurance_submission_v001.parquet'

# Save with the same column order as the sample, no index
submission[sub_template.columns.tolist()].to_parquet(parquet_path, index=False)

# Read both back from disk
sample_df = pd.read_parquet(sample_path)
sub_pq = pd.read_parquet(parquet_path)
sub_csv = pd.read_csv(out_path, dtype={'row_id': str})

sample_schema = pq.read_schema(sample_path).remove_metadata()
sub_schema = pq.read_schema(parquet_path).remove_metadata()

checks = {
    'parquet schema matches sample': sub_schema.equals(sample_schema),
    'column names and order match': list(sub_pq.columns) == list(sample_df.columns),
    'pandas dtypes match': sub_pq.dtypes.equals(sample_df.dtypes),
    'row count matches (626,639)': len(sub_pq) == len(sample_df) == 626_639,
    'row_ids identical and in same order': sub_pq['row_id'].equals(sample_df['row_id']),
    'no nulls': bool(sub_pq.notna().all().all()),
    'no negative predictions': bool((sub_pq['prediction'] >= 0).all()),
    # CSV stores floats as text, so allow for float parsing round-off (max diff printed below)
    'parquet data matches CSV': bool(sub_pq['row_id'].equals(sub_csv['row_id'])
                                     and np.allclose(sub_pq['prediction'], sub_csv['prediction'], rtol=1e-12, atol=0)),
}
display(pd.DataFrame({'check': list(checks), 'passed': list(checks.values())}))
print('Sample schema:    ', sample_schema)
print('Submission schema:', sub_schema)
print('Max |parquet - CSV| prediction difference:', (sub_pq['prediction'] - sub_csv['prediction']).abs().max())
assert all(checks.values()), 'Submission does not match the sample format'
print(f'Saved {parquet_path}')

### Submission v001 parquet: check results

`submissions/home_insurance_submission_v001.parquet` **exactly matches the format of `sample_submission.parquet`**. Only the predictions differ, and they are ours:

| Check | Result |
| --- | --- |
| Parquet schema (`row_id: string`, `prediction: double`) | Identical to sample |
| Column names and order | Identical |
| pandas dtypes | Identical |
| Row count | 626,639 in both |
| `row_id`s and row order | Identical |
| Nulls / negative predictions | None |
| Parquet vs CSV | Same `row_id`s; predictions identical (both rounded to 2 decimal places) |

Predictions differ from the sample as intended (the sample is all 0). Predictions are float64 rounded to 2 decimal places (the CSV writes every value as e.g. `342.89` or `0.00`). Buildings rows range from 1.13 to 420.28 CU (total 63.1M CU), and motor and contents rows are 0. An independent re-read of both files with `pyarrow` confirmed the schema, row count and row order match.

In [0]:
display(kw_data['sample_submission'].head(20))

In [0]:
# Save sample_submission as CSV (no index) alongside the parquet in data/
sample_csv_path = '../data/sample_submission.csv'
kw_data['sample_submission'].to_csv(sample_csv_path, index=False, float_format='%.2f')

# Check: re-read and compare with the parquet original
sample_csv = pd.read_csv(sample_csv_path, dtype={'row_id': str})
assert list(sample_csv.columns) == ['row_id', 'prediction']
assert sample_csv['row_id'].equals(kw_data['sample_submission']['row_id'])
assert np.allclose(sample_csv['prediction'], kw_data['sample_submission']['prediction'])
print(f'Saved {len(sample_csv):,} rows to {os.path.abspath(sample_csv_path)}')
with open(sample_csv_path) as f:
    print(''.join([next(f) for _ in range(3)]))